### Récupération des données météorologiques sur l'API OpenWeatherMap

Particularités techniques de l’API OpenWeather (plan gratuit)

Le code utilise le fichier produit par le notebook précédent (appel à Nominatim) pour récupérer les coordonnées GPS de chaque site — données obligatoires pour interroger OpenWeather One Call.

Avec le plan gratuit, la limite est d’environ 60 requêtes/minute, d’où le choix d’attendre 1 seconde entre chaque requête pour rester en dessous du seuil.

L’API peut parfois renvoyer une réponse sans le bloc daily (quota atteint ou éventuelle restriction du plan).
→ Dans ce cas, le site est simplement ignoré pour éviter les erreurs.

Certains champs sont optionnels :
- rain peut être absent → traité comme 0;
- pop est une probabilité.

In [ ]:
import requests, time, math
import pandas as pd
import plotly.express as px
from dotenv import load_dotenv
from pathlib import Path
import os

# --- Charger .env depuis le dossier projet (parent du notebook) ---
BASE_DIR = Path().resolve().parent   # va du notebook vers Project2-Kayak
load_dotenv(BASE_DIR / ".env")

# Charger le fichier généré avant avec Nominatim
file_path = BASE_DIR / "01-Get coordinates" / "sites_lon_lat.csv"
df_sites = pd.read_csv(file_path)  # colonnes attendues: site, lat, lon

url = "https://api.openweathermap.org/data/3.0/onecall"

# Récupérer la clé depuis .env
OPENWEATHER_API_KEY = os.getenv("OPENWEATHER_API_KEY")
if not OPENWEATHER_API_KEY:
    raise ValueError("OPENWEATHER_API_KEY manquant dans .env")

records = []  # lignes jour par jour
summary = []  # résumé par site

# --- 1) Boucle d'appel à One Call ---
for row in df_sites.itertuples(index=False):
    site, lat, lon = row

    if pd.isna(lat) or pd.isna(lon):
        summary.append({
            "id": len(summary) + 1,
            "site": site,
            "lat": lat, "lon": lon,
            "avg_temp_c": None,
            "expected_rain_mm": None,
            "nice_score": None
        })
        continue

    params = {
        "lat": float(lat),
        "lon": float(lon),
        "appid": OPENWEATHER_API_KEY,
        "units": "metric",
        "lang": "fr",
        "exclude": "minutely,hourly,alerts"
    }

    r = requests.get(url, params=params)
    data = r.json()

    daily = data.get("daily", [])
    if not isinstance(daily, list) or len(daily) == 0:
        summary.append({"id": len(summary)+1, "site": site,
                        "lat": lat, "lon": lon,
                        "avg_temp_c": None, "expected_rain_mm": None,
                        "nice_score": None})
        time.sleep(1)
        continue

    days = daily[:7]

    for d in days:
        dt = pd.to_datetime(d.get("dt", None), unit="s", utc=True).tz_convert("Europe/Paris") if d.get("dt") else None
        temp_day = (d.get("temp", {}) or {}).get("day", None)
        pop = d.get("pop", 0) or 0
        rain = d.get("rain", 0) or 0

        records.append({
            "site": site,
            "lat": float(lat),
            "lon": float(lon),
            "date": dt.date() if dt is not None else None,
            "temp_day_c": temp_day,
            "pop": pop,
            "rain_mm": rain
        })

    # calcul des métriques sur 7 jours
    df_tmp = pd.DataFrame(records[-len(days):])
    avg_temp_c = df_tmp["temp_day_c"].mean()
    expected_rain_mm = (df_tmp["pop"] * df_tmp["rain_mm"]).sum()
    nice_score = avg_temp_c - 0.5 * expected_rain_mm

    summary.append({
        "id": len(summary)+1,
        "site": site,
        "lat": float(lat),
        "lon": float(lon),
        "avg_temp_c": round(avg_temp_c, 2) if avg_temp_c is not None and not math.isnan(avg_temp_c) else None,
        "expected_rain_mm": round(expected_rain_mm, 2),
        "nice_score": round(nice_score, 2) if nice_score is not None else None
    })

    time.sleep(1)  # pour n'envoyer qu'une requête par seconde

# --- 2) DataFrames & sauvegarde ---
df_daily = pd.DataFrame(records)
df_summary = pd.DataFrame(summary)

df_daily.to_csv("sites_weather_daily.csv", index=False)
df_summary.to_csv("sites_weather_summary.csv", index=False)

print("Fichiers enregistrés : sites_weather_daily.csv et sites_weather_summary.csv")

In [8]:
# --- 3) Préparer les données pour Plotly ---
# Forcer les colonnes numériques
for col in ["avg_temp_c", "expected_rain_mm", "nice_score", "lat", "lon"]:
    df_summary[col] = pd.to_numeric(df_summary[col], errors="coerce")

# Garder seulement les sites valides, classer par nice_score et prendre le Top 5
df_top = (
    df_summary
    .dropna(subset=["nice_score", "lat", "lon", "avg_temp_c"])
    .sort_values("nice_score", ascending=False)
    .head(5)
    .copy()
)

if df_top.empty:
    print("Pas de sites avec données météo valides.")
else:
    # Taille positive pour l'affichage
    df_top["size_marker"] = (df_top["nice_score"] - df_top["nice_score"].min()) + 1
    # --- 4) Carte Plotly ---
    fig = px.scatter_mapbox(
        df_top,
        lat="lat",
        lon="lon",
        hover_name="site",
        hover_data={
            "avg_temp_c": True,
            "expected_rain_mm": True,
            "nice_score": True,
            "lat": False, "lon": False, "id": False
        },
        size="size_marker",
        color="avg_temp_c",                      # <— couleurs = température
        color_continuous_scale=[(0, "blue"), (0.5, "purple"), (1, "red")],
        labels={"avg_temp_c": "day_temperature"} # nom dans la légende
    )

    fig.update_layout(
        mapbox_style="open-street-map",
        mapbox_center={"lat": 46.6, "lon": 2.4},  # centré sur la France
        mapbox_zoom=4.7,
        margin={"r":0, "t":40, "l":0, "b":0},
        height=700, width=700,                    # carré
        title="Top 5 destinations (nice_score) – couleur = température jour (°C)"
    )
    fig.show()
